In [3]:
import pandas as pd
import numpy as np

# 1. 假设已读取原始高并发 transaction 数据（包含 user_id 和 timestamp）
# 这里我们直接利用时间戳将其转换为日期和周
df_raw = pd.read_csv('user_churn_data.csv') 
# 执行滑窗计算：
df_raw['date'] = pd.to_datetime(df_raw['recency_hours'], unit='h') # 模拟时间对齐

# 2. Step 1: 定义每个用户的首次活跃时间 (First Active Date/Week)
user_first_week = df_raw.groupby('user_id')['date'].min().dt.to_period('W').reset_index()
user_first_week.columns = ['user_id', 'cohort_week']

# 3. Step 2 & 3: 合并回原表计算周期差并生成 Cohort 矩阵数据
df_cohort = pd.merge(df_raw, user_first_week, on='user_id')
df_cohort['active_week'] = df_cohort['date'].dt.to_period('W')
df_cohort['week_index'] = (df_cohort['active_week'] - df_cohort['cohort_week']).apply(lambda x: x.n)

cohort_matrix = df_cohort.groupby(['cohort_week', 'week_index'])['user_id'].nunique().unstack().fillna(0)
cohort_size = cohort_matrix.iloc[:, 0]
cohort_retention = cohort_matrix.divide(cohort_size, axis=0)

print("--- 工业级 Cohort 留存矩阵已生成 ---")
print(cohort_retention.round(4) * 100)

--- 工业级 Cohort 留存矩阵已生成 ---
week_index                 0
cohort_week                 
1969-12-29/1970-01-04  100.0
1970-01-05/1970-01-11  100.0


In [4]:
from scipy import stats

# 模拟 2000 位高风险流失用户的实验数据 (0=未留存, 1=已留存)
# A组 (Control): 不发券，基准留存率约 21%
group_A = np.random.binomial(1, 0.21, 1000)
# B组 (Treatment): 发券，由于策略干预，留存率提升至 29%
group_B = np.random.binomial(1, 0.29, 1000)

# 执行双样本 t 检验
t_stat, p_value = stats.ttest_ind(group_A, group_B)

print(f"--- A/B 实验分析报告 ---")
print(f"Control Group (A) Retention Rate: {group_A.mean():.2%}")
print(f"Treatment Group (B) Retention Rate: {group_B.mean():.2%}")
print(f"T-Statistic: {t_stat:.4f} | P-Value: {p_value:.4e}")

if p_value < 0.05:
    print("📢 Business Conclusion: Voucher intervention significantly improved 7-day retention among high-risk users (p < 0.05). Proceed with full roll-out.")
else:
    print("📢 Business Conclusion: No statistically significant difference observed. Insufficient evidence to roll out the voucher policy.")

--- A/B 实验分析报告 ---
Control Group (A) Retention Rate: 21.50%
Treatment Group (B) Retention Rate: 28.80%
T-Statistic: -3.7737 | P-Value: 1.6553e-04
📢 Business Conclusion: Voucher intervention significantly improved 7-day retention among high-risk users (p < 0.05). Proceed with full roll-out.
